# revise：双固定效应诊断稿

这个 notebook 用于做一版独立的探索性诊断：

- 因变量使用 `AMR_AGG`
- 对 `X` 与 `Y` 做 `log1p(abs())`
- 主模型使用双固定效应（Province FE + Year FE）
- 同时给出 `Pooled OLS + dummies` 的整体 R² 作为对照

它不是当前主线 `fixed_effects_master_time.ipynb` 的直接替代版本，更适合作为方法诊断或附录草稿。


In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

# ==========================================
# 1. 数据读取与主键整理
# ==========================================
amr_path = r"C:\Users\lunch\Downloads\amr_rate.csv"
x_path = r"C:\Users\lunch\Downloads\climate_social_eco.csv"

amr = pd.read_csv(amr_path, encoding="utf-8-sig")
x = pd.read_csv(x_path, encoding="utf-8-sig")

# 为避免中英文列名混杂，这里显式固定 climate 表前两列为 Province / Year
x = x.rename(columns={x.columns[0]: "Province", x.columns[1]: "Year"})

def normalize_key_cols(df):
    col_map = {}
    for c in df.columns:
        cc = str(c).strip().lower()
        if cc in ["province", "prov", "省份"]:
            col_map[c] = "Province"
        if cc in ["year", "yr", "年份"]:
            col_map[c] = "Year"
    return df.rename(columns=col_map)

amr = normalize_key_cols(amr)
x = normalize_key_cols(x)

for df_temp in [amr, x]:
    df_temp["Province"] = df_temp["Province"].astype(str).str.strip()
    df_temp["Year"] = pd.to_numeric(df_temp["Year"], errors="coerce").astype("Int64")

df = amr.merge(x, on=["Province", "Year"], how="inner")
df = df[df["Year"].between(2014, 2023)].dropna(subset=["Province", "Year"]).copy()

# ==========================================
# 2. 变量定义：这一版是诊断稿，不与主线强行保持同一口径
# ==========================================
X_cols = ["PM2.5", "省平均气温", "医疗水平", "抗菌药物使用强度", "GDP", "城市用水普及率", "生活垃圾无害化处理率", "省平均降水", "R1xday"]
Y_col = "AMR_AGG"
RESULT_PATH = "results/revise_two_way_log_diagnostic.csv"
SUMMARY_PATH = "results/revise_two_way_log_summary.csv"

AMR_COLS = ["MRCNS", "VREFS", "VREFM", "PRSP", "ERSP", "3GCRKP", "MRSA", "3GCREC", "CREC", "QREC", "CRPA", "CRKP", "CRAB"]
df[Y_col] = df[AMR_COLS].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# 先做数值化，再按省份中位数 + 全局中位数补缺
for c in X_cols + [Y_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df.groupby("Province")[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())

# 这版保留 log1p(abs()) 设定，仅用于探索“对数化 + 双固定效应”的表现
df_log = df.copy()
for col in X_cols + [Y_col]:
    df_log[col] = np.log1p(df_log[col].abs())

df_log = df_log.replace([np.inf, -np.inf], np.nan).dropna(subset=X_cols + [Y_col])
df_log = df_log.set_index(["Province", "Year"]).sort_index()

# ==========================================
# 3. 多重共线性诊断
# ==========================================
def check_vif(X):
    X_clean = X.dropna()
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_clean.columns
    vif_data["VIF"] = [variance_inflation_factor(X_clean.values, i) for i in range(len(X_clean.columns))]
    return vif_data

vif_table = check_vif(df_log[X_cols])
print("--- 多重共线性检查 (VIF) ---")
print(vif_table)

# ==========================================
# 4. 模型运行
# ==========================================
model_fe = PanelOLS(df_log[Y_col], df_log[X_cols], entity_effects=True, time_effects=True)
res_fe = model_fe.fit(cov_type="clustered", cluster_entity=True)

print("\n--- 方案 A: PanelOLS (双固定效应诊断) ---")
print(f"R2 within:  {res_fe.rsquared_within:.4f}")
print(f"R2 overall: {res_fe.rsquared_overall:.4f}")

df_dummy = df_log.reset_index()
X_with_dummies = pd.get_dummies(
    df_dummy[["Province", "Year"] + X_cols],
    columns=["Province", "Year"],
    drop_first=True,
)
X_with_dummies = sm.add_constant(X_with_dummies.astype(float))

df_dummy_indexed = df_dummy.set_index(["Province", "Year"])
X_with_dummies.index = df_dummy_indexed.index

X_final = X_with_dummies.dropna()
Y_final = df_dummy_indexed[Y_col].loc[X_final.index]

model_pooled = PooledOLS(Y_final, X_final)
res_pooled = model_pooled.fit(cov_type="clustered", cluster_entity=True)

print("\n--- 方案 B: Pooled OLS + 虚拟变量（仅作 R2 对照） ---")
print(f"整体 R2: {res_pooled.rsquared:.4f}")

# ==========================================
# 5. 结果导出
# ==========================================
output_table = pd.DataFrame({
    "Predictor": res_fe.params.index,
    "Coefficient": res_fe.params.values,
    "p-value": res_fe.pvalues,
    "Lower CI": res_fe.conf_int().iloc[:, 0],
    "Upper CI": res_fe.conf_int().iloc[:, 1],
})

summary_table = pd.DataFrame([
    {"Metric": "nobs", "Value": int(res_fe.nobs)},
    {"Metric": "r2_within", "Value": float(res_fe.rsquared_within)},
    {"Metric": "r2_overall", "Value": float(res_fe.rsquared_overall)},
    {"Metric": "pooled_r2", "Value": float(res_pooled.rsquared)},
    {"Metric": "max_vif", "Value": float(vif_table["VIF"].max())},
    {"Metric": "median_vif", "Value": float(vif_table["VIF"].median())},
])

os.makedirs("results", exist_ok=True)
output_table.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
summary_table.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("\n分析完成。")
print("Detailed coefficients:", RESULT_PATH)
print("Diagnostic summary:", SUMMARY_PATH)


--- 多重共线性检查 (VIF) ---
      feature          VIF
0       PM2.5    20.055063
1       省平均气温    63.207848
2        医疗水平  1160.547039
3    抗菌药物使用强度   775.273797
4         GDP   310.511699
5     城市用水普及率  5818.494666
6  生活垃圾无害化处理率  5032.527747
7       省平均降水   348.380359
8      R1xday    72.538482

--- 方案 A: PanelOLS (双固定效应诊断) ---
R2 within:  -0.0851
R2 overall: 0.4581

--- 方案 B: Pooled OLS + 虚拟变量（仅作 R2 对照） ---
整体 R2: 0.9218

分析完成。
Detailed coefficients: results/revise_two_way_log_diagnostic.csv
Diagnostic summary: results/revise_two_way_log_summary.csv
